### Data Reading from CSV (Volumes)

In [0]:
df = spark.read.format("csv").option('inferSchema', True)\
        .option('header', True)\
        .load('/Volumes/deltalake/default/raw/BigMart Sales.csv')

df.display()


## Data Reading JSON

In [0]:
df_json = spark.read.format("json").option('inferSchema', True)\
                    .option('header', True)\
                    .option('multiline', False)\
                    .load('/Volumes/deltalake/default/raw/drivers.json')

df_json.display()


## SCHEMA - DDL 
## Method 1 : DDL Schema

In [0]:
df.printSchema()

In [0]:
my_ddl_schema = '''
        Item_Identifier string,
        Item_Weight string,
        Item_Fat_Content string,
        Item_Visibility double,
        Item_Type string,
        Item_MRP double,
        Outlet_Identifier string,
        Outlet_Establishment_Year integer,
        Outlet_Size string,
        Outlet_Location_Type string,
        Outlet_Type string,
        Item_Outlet_Sales double
'''
df = spark.read.format('csv')\
                .schema(my_ddl_schema)\
                .option('header', True)\
                .load('/Volumes/deltalake/default/raw/BigMart Sales.csv')
df.display()

### Method 2 : StructType() Schema

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [0]:
my_struct_schema = StructType([
    StructField('Item_identifier', StringType(), True),
    StructField('Item_Weight', StringType(), True),
    StructField('Item_Fat_Content', StringType(), True),
    StructField('Item_Visibility', StringType(), True),
    StructField('Item_Type', StringType(), True),
    StructField('Item_MRP', StringType(), True),
    StructField('Outlet_Identifier', StringType(), True),
    StructField('Outlet_Establishment_Year', StringType(), True),
    StructField('Outlet_Size', StringType(), True),
    StructField('Outlet_Location_Type', StringType(), True),
    StructField('Outlet_Type', StringType(), True),
    StructField('Item_Outlet_Sales', StringType(), True)
])

df = spark.read.format('csv')\
                .schema(my_struct_schema)\
                .option('header', True)\
                .load('/Volumes/deltalake/default/raw/BigMart Sales.csv')
                
df.display()


## **SELECT**
## Method 1

In [0]:
df_sel = df.select('Item_Identifier', 'Item_Weight', 'Item_Fat_Content').display()

## **Method 2** : using col 


In [0]:
df.select(col('Item_Identifier'), col('Item_Weight'), col('Item_Fat_Content')).display()

## Method 3 : SelectExpr

In [0]:
df.selectExpr('Item_Identifier', 'Item_Weight', 'Item_Fat_Content').display()

## Alias 

In [0]:
df.select(col('Item_Identifier').alias('Item_ID'), 
          col('Item_Weight').alias('Item_Wt'), 
          col('Item_Fat_Content').alias('Fat_Content')).display()

In [0]:
# df.select(col('Item_Weight').cast(DoubleType())).display()

df_filtered = df.where(col('Item_Weight').cast(DoubleType()) > 10)
display(df_filtered)

In [0]:
# where/filter transformation is used to filter rows based on a condition
# Both where() and filter() are equivalent in PySpark

# Example: Filter rows where Item_Weight > 10
df_filtered = df.where(col('Item_Weight').cast(DoubleType()) > 10)
display(df_filtered)

# Alternatively, using filter()
df_filtered2 = df.filter(col('Item_Weight').cast(DoubleType()) > 10)
# display(df_filtered2)

# Example: Apply two filters - Item_Weight > 10 and Item_Fat_Content == 'Low Fat'
df_filtered_multi = df.where((col('Item_Weight').cast(DoubleType()) > 10) & (col('Item_Fat_Content') == 'Low Fat'))
display(df_filtered_multi)

# Alternatively, using filter() with two conditions
df_filtered_multi2 = df.filter((col('Item_Weight').cast(DoubleType()) > 10) & (col('Item_Fat_Content') == 'Low Fat'))
# display(df_filtered_multi2)

# Senario 3 : Outlet_Location_Type = Tier1 or Tier 2 and OutletSize = null records
df_filtered3 = df.where(col('Outlet_Location_Type').isin(['Tier 1', 'Tier 2']) & col('Outlet_Size').isNull())
display(df_filtered3)



## **Rename Columns** : withColumnRenamed()

In [0]:
df.withColumnRenamed('Item_Weight', 'ItemWt').display()
# Note it changes column name at Dataframe level

## **_Create a new column_** : withColumn

In [0]:
## Senario 1 : Create a new constant column
# To add any constant value we use lit() function
df.withColumn('Flag', lit('new')).display()

In [0]:
# Senario 2
# Create a new column with multiple of Item_Wight and Item_MRP
df.withColumn('multiply', col('Item_Weight').cast(DoubleType())*col('Item_MRP')).display()

In [0]:
# Senario 2 : Replace low fat with LF and Regular with Reg
df.withColumn('Item_Fat_Content', regexp_replace(col('Item_Fat_Content'), 'Regular', 'Reg'))\
    .withColumn('Item_Fat_Content', regexp_replace(col('Item_Fat_Content'), 'Low Fat', 'LF'))\
    .withColumn('Item_Fat_Content', regexp_replace(col('Item_Fat_Content'), 'low fat', 'LF')).display()


## Type Casting 

In [0]:
df.withColumn('Item_Weight', col('Item_Weight').cast(DoubleType())).display()

## Sort/Order By : .sort(col(<col_name>).desc()), .sort(col(<col_name>).desc())

In [0]:
df.sort(col('Item_weight').desc()).display()

In [0]:
df.sort(col('Item_Visibility').asc()).display()

In [0]:
# Sorting based on multiple columns and both of them are in descending order
df.sort(['Item_Weight', 'Item_Visibility'], ascending=[0, 0]).display()

In [0]:
# Sorting based on multiple columns and both of them are in descending order
df.sort(['Item_Weight', 'Item_Visibility'], ascending=[0, 1]).display()

## Limit

In [0]:
df.limit(10).display()

## DROP
### Senario 1 : Drop 1 column

In [0]:
df.drop('Item_Visibility').display()

In [0]:
# Scenario 2 : Drop multiple columns

df.drop(col('Item_Visibility'), col('Item_Type'), col('Item_MRP')).display()

## Drop Duplicates

In [0]:
# Drop duplicate rows from the dataframe
# This process is also called Dedup
# distinct() is also having the same functionality
df.dropDuplicates().display()

In [0]:
df.dropDuplicates(['Item_Identifier', 'Item_Type']).display()

In [0]:
df.drop_duplicates(subset=['Item_Type']).display()

### ## UNION AND UNION BYNAME![image_1771415645058.png](./image_1771415645058.png "image_1771415645058.png")

In [0]:
data1 = [('1', 'kad'), 
         ('2', 'kad')]
schema1 = 'id STRING, name STRING'
df1 = spark.createDataFrame(data1, schema1)
df1.display()

data2 = [('3', 'rahul'),
         ('4', 'jas')]
schema2 = 'id STRING, name STRING'
df2 = spark.createDataFrame(data2, schema2)
df2.display()


## UNION

In [0]:
df1.union(df2).display()

In [0]:
data1 = [('kad', '1'), 
         ('sid', '2')]
schema1 = 'name STRING, id STRING'
df1 = spark.createDataFrame(data1, schema1)
df1.display()

In [0]:
# Put values inside wrong column. To avoid this from happening we use UnionByName() function

df1.union(df2).display()

In [0]:
df1.unionByName(df2).display()

## String Functions 
## INITCAP()
## UPPER()
## LOWER()

In [0]:
df.select(initcap('Item_Type').alias('InitCap_Item_Type')).limit(4).display()  # first and letter after space in capital
df.select(lower('Item_Type')).limit(4).display() # all letters in lower case
df.select(upper('Item_Type')).limit(4).display() # all letters in upper case
df.select(translate('Item_Type', ' ', '_')).limit(4).display() # replace space with underscore

## Date Functions
Current_Date()
Date_Add()
Date_Sub()

In [0]:
# Current_Date()
df_new = df.withColumn('Curr_date', current_date())\
        .withColumn('Yesterday_date', date_add(current_date(), -1))\
        .withColumn('date_week_day', date_add(current_date(), 7))\
        .withColumn('date_diff', datediff(col('date_week_day'), col('Yesterday_date')))
df_new.display()

# Date_Add()

## Date_Format() method

In [0]:
df_new = df_new.withColumn('Yesterday_date', date_format('Yesterday_date', 'dd-MM-yyyy'))
df_new.display()

### Different ways to handle nulls inside Pyspark.
Category 1 : Dropping null values
Category 2 : Filling null values

In [0]:
# Handling Nulls => Dropping Nulls
df.dropna('all').display()  # it will drop nulls which have nulls in all the columns 

In [0]:
# This is very sensitive to use since all the rows are getting removed.
df.dropna('any').display()  # it will drop nulls in any of the columns

In [0]:
# Last way is to drop nulls from a subset of columns.
df.dropna(subset=['Item_Weight']).display()

## Fill Null Values

In [0]:
# Scenario 1: Fill the null values in all the columns
df.fillna('NotAvailable').display()

# Scenario 2: Fill the null values only in specific column
df.fillna('Not_Available', subset=['Outlet_Size']).display()

## Advanced Function SPLIT and Indexing 
(Time : 2:45)
![image_1771420607722.png](./image_1771420607722.png "image_1771420607722.png")

In [0]:
df.withColumn('Outlet_Type', split('Outlet_Type', ' ')).display()

## Indexing 

In [0]:
# Indexing 

df.withColumn('Outlet_Type', split('Outlet_Type', ' ')[1]).display()

## Explode : It creates a single row for each index inside the list present inside the column
![image_1771420946409.png](./image_1771420946409.png "image_1771420946409.png")

## Explode

In [0]:
df_exp = df.withColumn('Outlet_Type', split('Outlet_Type', ' '))
df_exp2 = df.withColumn('Outlet_Type', split('Outlet_Type', ' '))
df_exp.display()

In [0]:
df_exp = df_exp.withColumn('Outlet_Type', explode('Outlet_Type'))
df_exp.display()


## Array_Contains
### Used for creating custom columns based on conditions we are using.
3:00

In [0]:
df_exp2.withColumn('Type1_flag', array_contains('Outlet_Type', 'Type1')).display()

## Group_By
### Senario - 1

In [0]:
df.display()
df.describe

In [0]:
# Find aggregation based on Item_Type and sum of Item_MRP

df.groupBy('Item_Type') \
  .agg(sum(col('Item_MRP')).alias('Total_MRP')) \
  .filter("Total_MRP > 1000") \
  .display()

In [0]:
df.groupBy('Item_type')\
    .agg(avg(col('Item_MRP').alias('Item_MRP_Avg')))\
    .display()

In [0]:
df.groupBy(['Item_type', 'Outlet_Size'])\
    .agg(sum('Item_MRP').alias('Item_MRP_Sum'))\
    .display()

In [0]:
# Senario : group by multiple columns and aggreagation by multiple columns.

df.groupBy(['Item_type', 'Outlet_Size'])\
    .agg(sum('Item_MRP').alias('Item_MRP_Sum'), avg('Item_MRP').alias('Item_MRP_Avg'))\
    .display()

## Collect_List

In [0]:
data = [('user1', 'book1'),
         ('user1', 'book2'),
         ('user2', 'book2'),
         ('user2', 'book4'),
         ('user3', 'book5'),
         ('user4', 'book3')]

schema = 'user string, book string'
df_book = spark.createDataFrame(data, schema)
df_book.display()

In [0]:
# It is similar to group concat inside mysql.

df_book.groupBy('user').agg(collect_list('book')).display()


# **PIVOT**

In [0]:
df.groupBy('Item_Type').pivot('Outlet_Size').agg(avg('Item_MRP')).display()

## WHEN-OTHERWISE
###  When we want to build conditional columns make use of the function When-Otherwise.

In [0]:
# Senario 1
df.withColumn('veg_flag', when(col('Item_Type')=='Meat', 'Non-Veg').otherwise('Veg')).display()

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

# Senario 1
df_flag = df.withColumn('veg_flag', when(col('Item_Type')=='Meat', 'Non-Veg').otherwise('Veg'))

# Senario 2
df_flag.withColumn('veg_exp_flag', when((col('veg_flag')=='Veg') & (col('Item_MRP') < 100), 'Veg_Inexpensive')\
                            .when((col('veg_flag')=='Veg') & (col('Item_MRP') > 100), 'Veg_Expensive')\
                            .otherwise('Veg')).display()


## JOINs
### 1. INNER JOIN
### 2. LEFT JOIN
### 3. RIGHT JOIN
### 4. FULL JOIN
### 5. ANTI JOIN

In [0]:
data1 = [('1', 'gaur', 'd01'),
         ('2', 'kit', 'd02'),
         ('3', 'aka', 'd03'),
         ('4', 'giri', 'd04'),
         ('7', 'sayo', 'd07'),
         ('6', 'kishan', 'd06')]

schema1 = 'emp_id STRING, emp_name STRING, dept_id STRING'

df1 = spark.createDataFrame(data1, schema1)

data2 = [('d01', 'HR'), 
        ('d02', 'Maketing'), 
        ('d03', 'Accounts'), 
        ('d04', 'IT'), 
        ('d05', 'Finance')]

schema2 = 'dept_id STRING, department STRING'

df2 = spark.createDataFrame(data2, schema2)

df1.display()
df2.display()


In [0]:
df1.join(df2, df1['dept_id'] == df2['dept_id'], 'inner').display()

In [0]:
df1.join(df2, df1['dept_id'] == df2['dept_id'], 'left').display()

In [0]:
df1.join(df2, df1['dept_id'] == df2['dept_id'], 'right').display()

In [0]:
df1.join(df2, df1['dept_id'] == df2['dept_id'], 'anti').display()

## Window Functions
### Row Number() :
Use cases : 
1. To generate a surevey
2. To remove duplicates
3. Used to generate unique records


In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

window = Window().orderBy('Item_Identifier')
df.withColumn('rowCol', row_number().over(window)).display()

## Window Functions
### RANK() and DENSE_RANK()
Use cases : 
1. Whenever we are going to provide ranking to our dataframe

RANK()  
a, 1
a, 1
b, 3
b, 3
b, 3
c, 6

DenseRank()
a, 1
a, 1
b, 2
b, 2
b, 2
c, 3

In [0]:
# Window functions
from pyspark.sql.window import Window
from pyspark.sql.functions import rank, dense_rank, row_number

df.withColumn('rank', rank().over(Window.orderBy(col('Item_Identifier').desc())))\
    .withColumn('denseRank', dense_rank().over(Window.orderBy(col('Item_Identifier').desc())))\
    .withColumn('rowCol', row_number().over(Window.orderBy(col('Item_Identifier').desc())))\
    .display()

Windows Functions 
## Cumulative Sum
Means we are have to add all the values row by row one after the other. 
No  |  CumSum
1   |  1
2   |  3
4   |  7
15  |  22
21  |  43
    |

### This is a fusion of sum function with window function


In [0]:
df.withColumn('cumsum', sum(col('Item_MRP')).over(Window.orderBy('Item_Type'))).display()

In [0]:
# Adding the frame clause
df.withColumn('cumsum', sum(col('Item_MRP')).over(Window.orderBy('Item_Type').rowsBetween(Window.unboundedPreceding, Window.currentRow))).display()

In [0]:
# Adding the frame clause
df.withColumn('cumsum', sum(col('Item_MRP')).over(Window.orderBy('Item_Type').rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing))).display()

## USER DEFINED FUNCTIONS
### It is recommended not to create User Defined Functions. Lot of computation is lost while creating User Defined functions in Python or pyspark.
Since Executor compiler runs on Java Virtual Machine (JVM). Say you have created a user defined fucntion and in reality you are writing your code in python. They do not support python. They need to add python enterpretor. So first it will be computed in python then it will covert the code in JVM and then execute it. This will consume lot of computation power. This the reason why we avoid User Defined function. 

In [0]:
## USER DEFINED FUNCTIONS
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

def my_func(x):
    return x*x


In [0]:
my_udf = udf(my_func)

In [0]:
df.withColumn('myNewCol', my_udf('Item_MRP')).display()

## Data Writing 


### CSV

In [0]:
df.write.format('csv')\
    .save('/Volumes/deltalake/default/datalake/data.csv')

### Data Writing Modes :
1. Append : It will send the copy or any dataframe we have chosen there. If the folder has data it will add another file.
2. Overwrite : In this case it will delete the old file and add a new one. in this senario we loose the old information. So use it causiously.
3. Ignore : If say a file exists and we don't want it to overwrite it or throw an error it will just ignore it.
4. ErrorIfExists : If don't want to update the same location then we can set it up in a way so that it won't throw an error. 


In [0]:
# Append
df.write.format('csv')\
    .mode('append')\
    .save('/Volumes/deltalake/default/datalake/data.csv')

# Alternative method
df.write.format('csv')\
    .mode('append')\
    .option('path','/Volumes/deltalake/default/datalake/data.csv')\
    .save()



In [0]:
# Overwrite
df.write.format('csv')\
    .mode('overwrite')\
    .option('path','/Volumes/deltalake/default/datalake/data.csv')\
    .save()

In [0]:
# Error
df.write.format('csv')\
    .mode('error')\
    .option('path','/Volumes/deltalake/default/datalake/data.csv')\
    .save()

In [0]:
# Ignore
df.write.format('csv')\
    .mode('ignore')\
    .option('path','/Volumes/deltalake/default/datalake/data.csv')\
    .save()

### Parquet : 
Columnar file format. Files are pulled based on columns rather than being pulled on row basis. Standard pproad to use is case of Big Data. Inside parquet file hearder gets stored in the footer as meta data. 

## Delta : 
They are an enhanced version of Parquet file format. In this case header or meta data or regarding updates gets stored inside Delta log. This is the backbone of Delta lake and future file formats.

### AVRO : 
Helpful in OLTP database. Useful for CRUD operations. 

In [0]:
# Parquet 

df.write.format('parquet')\
    .mode('overwrite')\
    .option('path','/Volumes/deltalake/default/datalake/data.parquet')\
    .save()


In [0]:
# delta 

df.write.format('delta')\
    .mode('overwrite')\
    .option('path','/Volumes/deltalake/default/datalake/data2.parquet')\
    .save()

### Table
Table on top of csv or any file format.

In [0]:
# Table
df.write.format('delta')\
    .mode('overwrite')\
    .saveAsTable('my_table')

## Managed Vs External Tables
### Managed Table
In case of managed table we created data on top of data present inside Volume (or Data Lake). When we create any table on top of delta table it gets created inside Databricks default storage location also called Managed location or Managed Table. So in that senario that data that is being used to create the table is mananged by Databricks. So, if by mistake we deleted the table it will also automatically delete the underlying data because databricks will think we don't need this information.

### External Tables 
We want to create table on top on dataframe. This time data will be stored in our own location because I am owning this data so whenever I am performing any delete operation or any kind of update operation it will not drop this data. It will only drop the schema. But the external data will remain same and safe. This is the External Table. And here the backend data is managed my US.


In [0]:
df.display()

## SPARK SQL
### TEMP_VIEW
All the views created here for performing SQL are temp views and they will be eleminated as soon as the sesssion ends.


In [0]:
# Convert DataFrame into a SQL view for Spark SQL

df.createTempView('my_view')

In [0]:
%sql
Select * from my_view
where Item_Fat_Content = 'Low Fat'

In [0]:
# Converted my Spark SQL into a Pyspark Dataframe

df_sql = spark.sql("""
Select * from my_view
where Item_Fat_Content = 'Low Fat'
""")
df_sql.display()